# Generate Disease–Symptom Associations

Calls a locally hosted vLLM OpenAI-compatible server
(`http://127.0.0.1:8000`) to synthesize a corpus of
disease–symptom JSON records, one per request, and appends them to
`../outs/associations-generated-01102026.jsonl`.

The output of this notebook is the *input* to
`rag-sym-semantic-match-vllm.ipynb`.


## 1. Imports and JSON-repair helper

Small LLMs frequently return JSON wrapped in markdown fences or with
trailing commentary. `fix_json` strips that wrapper so `json.loads`
can consume the result.


In [ ]:
import json
import re
import requests


def fix_json(text: str) -> str:
    """Extract a single JSON object from a noisy LLM response.

    Handles common failure modes:
    - ```json ... ``` markdown fences
    - leading/trailing prose around the object
    - smart quotes substituted for straight quotes
    """
    if not isinstance(text, str):
        return ""
    # Drop markdown code fences if present.
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    # Normalize smart quotes back to ASCII.
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    # Keep only the first balanced {...} block.
    match = re.search(r"\{.*\}", text, flags=re.S)
    return match.group(0) if match else text.strip()


## 2. Single-association generator

One prompt → one association. The prompt nudges the model toward
*uncommon* diseases by including two few-shot examples and asking it
to vary categories. On any parsing/network failure we fall back to a
harmless placeholder so the generation loop never crashes.


In [ ]:
def generate_unique_association(model_name, shots):
    """Ask the local vLLM server for one disease/symptom JSON record."""
    base_prompt = f"""
        Generate ONE disease–symptom association.

        Requirements:
        - Output ONLY valid JSON.
        - No backticks or fences.
        - Choose a medically plausible disease and symptom.
        - Prefer uncommon or less typical diseases.
        - Vary disease categories.

        Examples:
        {shots[0]}
        {shots[1]}
    """

    # Local OpenAI-compatible endpoint exposed by vLLM.
    r = requests.post(
        "http://127.0.0.1:8000/v1/chat/completions",
        json={
            "model": model_name,
            "messages": [
                {"role": "system", "content": "You output strict JSON only."},
                {"role": "user", "content": base_prompt},
            ],
            "max_tokens": 128,
            "temperature": 0.8,
            "top_p": 0.95,
            "top_k": 60,
        },
    )

    text = r.json()["choices"][0]["message"]["content"].strip()
    cleaned = fix_json(text)

    try:
        assoc = json.loads(cleaned)
        disease = assoc.get("Disease", "").strip()
        symptom = assoc.get("Symptom", "").strip()
        if disease and symptom:
            return assoc
    except (json.JSONDecodeError, AttributeError):
        # Fall through to the placeholder so the outer loop keeps going.
        pass

    # Safe fallback — keeps record count stable for downstream alignment.
    return {
        "Disease": "Rare genetic disorder",
        "Symptom": "progressive muscle weakness",
    }


## 3. Run the generation loop

Generates `N` associations and keeps them in memory. With a 7B model
on a single GPU expect this to take a while; the heartbeat print
every 100 iterations lets you watch progress.


In [6]:
count = 0
generated = []
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Two diverse exemplars so the model has both an infectious and a
# chronic-disease anchor in its few-shot context.
shots = [
    '{"Disease": "Influenza", "Symptom": "fever"}',
    '{"Disease": "Diabetes mellitus", "Symptom": "excessive thirst"}',
]

for i in range(10_000):
    assoc = generate_unique_association(model_name, shots)
    generated.append(assoc)

    if i % 100 == 0:
        print(f"Current batch count: {i}")
        print("Current association generated:", assoc)


Current batch count: 0
Current association generated: {'Disease': 'Amyotrophic lateral sclerosis', 'Symptom': 'muscle cramps'}
Current batch count: 100
Current association generated: {'Disease': 'Primary hypothyroidism', 'Symptom': 'myxedema'}
Current batch count: 200
Current association generated: {'Disease': 'Lupus erythematosus', 'Symptom': 'photosensitivity'}
Current batch count: 300
Current association generated: {'Disease': 'Lyme disease', 'Symptom': 'arthritis'}
Current batch count: 400
Current association generated: {'Disease': 'Leiomyosarcoma', 'Symptom': 'abdominal mass'}
Current batch count: 500
Current association generated: {'Disease': 'Lupus erythematosus', 'Symptom': 'photosensitivity'}
Current batch count: 600
Current association generated: {'Disease': 'Amyloidosis', 'Symptom': 'peripheral neuropathy'}
Current batch count: 700
Current association generated: {'Disease': 'Primary hypothyroidism', 'Symptom': 'myxedema'}
Current batch count: 800
Current association generate

## 4. Persist the run

Appends (not overwrites) so multiple notebook runs can accumulate.


In [ ]:
with open("../outs/associations-generated-01102026.jsonl", "a") as f:
    for assoc in generated:
        if assoc is not None:
            f.write(json.dumps(assoc) + "\n")


In [ ]:
# Quick size check after a run:
# len(generated)
